Try how averaging logits over multiple frames changes results.

In [8]:
# Colab/local setup: mount Drive, set project root, add src to path
import sys, os

# Install deps if running on Colab
if 'google.colab' in sys.modules:
    try:
        import torch, torchvision, cv2  # noqa: F401
    except Exception:
        %pip -q install torch torchvision opencv-python tqdm ipywidgets
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/AML-ETH-Project2'
else:
    # Local run
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    %load_ext autoreload
    %autoreload 2

SRC_PATH = os.path.join(PROJECT_ROOT, 'src')
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

print('Project root:', PROJECT_ROOT)
print('Using src path:', SRC_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/AML-ETH-Project2
Using src path: /content/drive/MyDrive/AML-ETH-Project2/src


In [9]:
# Clear cached modules to force fresh imports (useful during iterative work)
import sys
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['utils', 'dataset', 'models', 'train', 'UNet', 'post_processing']):
        del sys.modules[mod]
print("✓ Module cache cleared")

✓ Module cache cleared


In [10]:
# Imports and load data
import numpy as np
import torch
import torch.nn.functional as F
import cv2
from utils.utilities import load_zipped_pickle
import os

DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'raw', 'test.pkl')
data = load_zipped_pickle(DATA_PATH)
print(f'Loaded {len(data)} samples from', DATA_PATH)

Loaded 20 samples from /content/drive/MyDrive/AML-ETH-Project2/data/raw/test.pkl


In [11]:
# Load UNet model weights
from models.UNet import UNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet(n_channels=1, n_classes=2)

weights_path = os.path.join(PROJECT_ROOT, 'weights', 'unet_augmented.pth')
state = torch.load(weights_path, map_location=device)
model.load_state_dict(state)
model = model.to(device).eval()

# Print basic info
param_count = sum(p.numel() for p in model.parameters())
print(f"✓ Model loaded from {weights_path} | params: {param_count:,}")

✓ Model loaded from /content/drive/MyDrive/AML-ETH-Project2/weights/unet_augmented.pth | params: 37,651,586


In [12]:
# Averaged-logits post-processing helper (batched + GPU resize)
TARGET_HW = (256, 256)  # (H, W) the model was trained on

def compute_averaged_masks_batched(model, video_np, device, temp_window=2, batch_size=32):
    """
    Batched inference with temporal averaging of logits.
    - Resizes frames to 256x256 using torch (GPU/CPU) for speed
    - Runs model in batches
    - Averages logits over a ±temp_window
    - Resizes predictions back to original size with nearest neighbor

    video_np: numpy array (H_orig, W_orig, T)
    returns: numpy array masks (T, H_orig, W_orig), dtype=uint8 in {0,1}
    """
    H0, W0, T = video_np.shape
    model.eval()

    # 1) All frames -> tensor (T,1,H0,W0) in [0,1]
    frames = torch.from_numpy(video_np.astype(np.float32) / 255.0)  # (H0,W0,T)
    frames = frames.permute(2, 0, 1).unsqueeze(1)  # (T,1,H0,W0)

    # 2) Resize on device to TARGET_HW using bilinear
    frames = frames.to(device, non_blocking=True)
    frames_256 = F.interpolate(frames, size=(TARGET_HW[0], TARGET_HW[1]), mode='bilinear', align_corners=False)  # (T,1,256,256)

    # 3) Inference in batches → collect logits (T,2,256,256)
    logits_list = []
    with torch.no_grad():
        for s in range(0, T, batch_size):
            e = min(T, s + batch_size)
            logits_b = model(frames_256[s:e])  # (B,2,256,256)
            logits_list.append(logits_b)
    logits_all = torch.cat(logits_list, dim=0)  # (T,2,256,256)

    # 4) Temporal averaging and upsample back to original size
    preds = []
    for t in range(T):
        s = max(0, t - temp_window)
        e = min(T, t + temp_window + 1)
        avg_logits = logits_all[s:e].mean(dim=0)          # (2,256,256)
        pred_256 = avg_logits.softmax(dim=0).argmax(0)    # (256,256)
        pred_orig = F.interpolate(
            pred_256.unsqueeze(0).unsqueeze(0).float(),
            size=(H0, W0),
            mode='nearest'
        ).squeeze().to(torch.uint8).cpu().numpy()
        preds.append(pred_orig)

    return np.stack(preds, axis=0)  # (T, H0, W0)

In [13]:
# Run averaged-logit inference on a few videos (batched)
num_videos = min(1, len(data))
results = []

# Parameters
TEMP_WINDOW = 2
BATCH_SIZE = 32

for i in range(num_videos):
    sample = data[i]
    video = np.asarray(sample['video'])  # (H,W,T)
    masks_avg = compute_averaged_masks_batched(model, video, device, temp_window=TEMP_WINDOW, batch_size=BATCH_SIZE)  # (T,H,W)
    results.append({
        'name': sample.get('name', f'sample_{i}'),
        'video_shape': video.shape,
        'masks_shape': masks_avg.shape,
        'masks': masks_avg,
    })
    print(f"[{i}] {results[-1]['name']}: video {video.shape} -> masks {masks_avg.shape} | window={TEMP_WINDOW}, batch={BATCH_SIZE}")

[0] E9AHVWGBUF: video (586, 821, 103) -> masks (103, 586, 821) | window=2, batch=32


In [14]:
# Visualize one sample with overlay
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import IntSlider

idx = 0  # which result to visualize
video = np.asarray(data[idx]['video'])
H, W, T = video.shape
masks = results[idx]['masks']  # (T,H,W)

slider = IntSlider(min=0, max=min(T-1, masks.shape[0]-1), value=0, description='frame')

def show_frame(t):
    plt.figure(figsize=(5,5))
    plt.imshow(video[..., t], cmap='gray')
    rgba = np.zeros((H, W, 4), dtype=np.float32)
    rgba[..., 0] = 1.0
    rgba[..., 3] = masks[t].astype(float) * 0.45
    plt.imshow(rgba, interpolation='nearest')
    plt.title(f"{results[idx]['name']} — frame {t}")
    plt.axis('off')
    plt.show()

widgets.interact(show_frame, t=slider)

interactive(children=(IntSlider(value=0, description='frame', max=102), Output()), _dom_classes=('widget-inter…

<function __main__.show_frame(t)>